# MT5 AI Trader ? Kaggle One-Click Auto Train

Ch?y **cell b?n d??i** l? ??: clone/pull repo, install deps, t?m CSV trong `/kaggle/input`, auto-improve, verify report, copy model/report, zip output.

Kh?ng ch?y MT5/paper/demo/live tr?n Kaggle.


In [ ]:
# Kaggle one-click auto train: run this cell only
import os, sys, glob, json, shutil, subprocess, platform

REPO_URL = "https://github.com/quoccuongphan300992-cmd/mt5-ai-trader.git"
REPO_DIR = "/kaggle/working/mt5-ai-trader"
SYMBOL = "EURUSD"
TIMEFRAME = "H1"
BARS = 50000
LOCAL_CSV = "data/EURUSD_H1.csv"

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Kaggle inputs:", glob.glob("/kaggle/input/*"))

# 1) Clone/update repo
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)
print("CWD:", os.getcwd())

# 2) Install dependencies
req = "requirements-colab.txt" if os.path.exists("requirements-colab.txt") else "requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)

# 3) Find/copy CSV from Kaggle Input
os.makedirs("data", exist_ok=True)
if not os.path.exists(LOCAL_CSV):
    candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    print("CSV candidates:")
    for path in candidates:
        print(" -", path)
    if not candidates:
        raise FileNotFoundError("No CSV found under /kaggle/input. Add your CSV dataset via Add Input.")
    preferred = [x for x in candidates if "EURUSD" in os.path.basename(x).upper() and "H1" in os.path.basename(x).upper()]
    source_csv = preferred[0] if preferred else candidates[0]
    shutil.copy2(source_csv, LOCAL_CSV)
    print("Using CSV:", source_csv)
print("LOCAL_CSV:", os.path.abspath(LOCAL_CSV), os.path.getsize(LOCAL_CSV), "bytes")

# 4) Continue-train existing promising candidate auto_0050
CANDIDATE_ID = "auto_0050"
CANDIDATE_MODEL_DIR = "/kaggle/working/mt5_ai_outputs/models/candidates"
OUTPUT_CANDIDATE_DIR = CANDIDATE_MODEL_DIR
os.makedirs(CANDIDATE_MODEL_DIR, exist_ok=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Find parent auto_0050 artifacts from previous Kaggle output added via Add Input.
# Expected examples:
# /kaggle/input/.../models/candidates/auto_0050/model.joblib
# /kaggle/input/.../mt5_ai_outputs/models/candidates/auto_0050/model.joblib
candidate_model_matches = glob.glob(f"/kaggle/input/**/{CANDIDATE_ID}/model.joblib", recursive=True)
if not candidate_model_matches:
    raise FileNotFoundError(
        f"Parent candidate {CANDIDATE_ID} model.joblib not found under /kaggle/input. "
        "Add previous mt5_ai_outputs_kaggle output as Kaggle Input first."
    )
PARENT_MODEL_PATH = candidate_model_matches[0]
PARENT_METADATA_PATH = os.path.join(os.path.dirname(PARENT_MODEL_PATH), "metadata.json")
if not os.path.exists(PARENT_METADATA_PATH):
    PARENT_METADATA_PATH = None
print("Parent candidate model:", PARENT_MODEL_PATH)
print("Parent candidate metadata:", PARENT_METADATA_PATH)

cmd = [
    sys.executable, "-u", "main.py", "continue-train-candidate",
    "--csv", LOCAL_CSV,
    "--symbol", SYMBOL,
    "--timeframe", TIMEFRAME,
    "--bars", str(BARS),
    "--candidate-id", CANDIDATE_ID,
    "--candidate-model-path", PARENT_MODEL_PATH,
    "--add-estimators", "300",
    "--candidate-model-dir", CANDIDATE_MODEL_DIR,
    "--output-dir", OUTPUT_CANDIDATE_DIR,
]
if PARENT_METADATA_PATH:
    cmd.extend(["--candidate-metadata-path", PARENT_METADATA_PATH])
print("Running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, env=os.environ.copy())

# 5) Verify continue-train manifest
continued_dirs = sorted(glob.glob(f"{OUTPUT_CANDIDATE_DIR}/{CANDIDATE_ID}_continued_*"))
if not continued_dirs:
    raise FileNotFoundError("continued candidate output missing")
continued_dir = continued_dirs[-1]
manifest_path = os.path.join(continued_dir, "continue_train_manifest.json")
if not os.path.exists(manifest_path):
    raise FileNotFoundError("continue_train_manifest.json missing; continue-train did not complete")
manifest = json.load(open(manifest_path, encoding="utf-8"))
print(json.dumps(manifest, indent=2))
print("continued_candidate_dir:", continued_dir)
print("train_mode:", manifest.get("train_mode"))

# 6) Copy and zip outputs for Kaggle download
out_dir = "/kaggle/working/output"
for sub in ["models", "reports"]:
    os.makedirs(f"{out_dir}/{sub}", exist_ok=True)
    if os.path.exists(sub):
        for item in glob.glob(f"{sub}/*"):
            dst = f"{out_dir}/{sub}/{os.path.basename(item)}"
            if os.path.isdir(item):
                shutil.copytree(item, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(item, dst)

zip_path = "/kaggle/working/mt5_ai_outputs_kaggle.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_path[:-4], "zip", out_dir)

print("=== OUTPUT FILES ===")
for path in sorted(glob.glob(f"{out_dir}/**/*", recursive=True)):
    if os.path.isfile(path):
        print(path)
print("ZIP:", zip_path, os.path.getsize(zip_path), "bytes")
